# 03 Feature matrix: checks, feature-set decisions and stress tests (slice 3)

Checks the matrix built by `src/prepare.py` and `src/features.py` (see `docs/data_dictionary.md`) and records the evidence behind the feature-set decisions: how every feature looks for benign and tunnel traffic, which benign queries come closest to the tunnel, how the tunnel phases differ, how far the test period drifts, which features are redundant, which feature set ranks best, and how the set holds up under slow tunnels and a noisier network.

Default run: `session="second"` (burst 2 is the tunnel session, burst 1 is removed from the stream), train before 18:30 UTC, 60 s window, features counted per (client, parent domain). The strict variant (`session="first"`, cut 18:04:36) is compared where stated.

**Default feature set (`feature_sets(60)["forced"]`):** `name_bytes`, `uniq_names_60s`, `uniq_ratio_60s`, `norepeat_tol`. Ablation columns (`extra`): `longest_label`, `entropy`, `norepeat_evidence`, `log_cum_uniq`. Record type is the optional `data_qtype`.

**Scope:** sections 6 to 9 use Isolation Forest only to compare *feature sets* (threshold-free ROC and PR AUC). They are not the detector evaluation; thresholds, metrics per phase and false-positive analysis belong to the modelling notebook.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.features import make_matrix, build_features, feature_sets, drift_table
from src.prepare import load_raw, prepare
from src.variants import add_benign_clients, collapse_clients, inject_repeats, slow_tunnel

FIG = ROOT / "notebooks" / "figures"
FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 200)

m = make_matrix(session="second", window_s=60, write=False)
m_first = make_matrix(session="first", window_s=60, write=False)
fs = feature_sets(60)
DEFAULT = fs["forced"]
FEATS = fs["all"]
print(m.shape)
print(m.groupby(["split", "label"]).size().unstack(fill_value=0))
print("missing values in features:", int(m[FEATS].isna().sum().sum()))
print("default set:", DEFAULT)
print("ablation columns:", fs["extra"])


def rank_scores(frame, cols, seeds=10):
    """Fit Isolation Forest on the training rows, score the test rows. Returns mean ROC AUC and PR AUC
    (threshold-free) over `seeds` random seeds."""
    tr, te = frame[frame.split == "train"], frame[frame.split == "test"]
    out = []
    for s in range(seeds):
        f = IsolationForest(n_estimators=300, random_state=s).fit(tr[cols])
        score = -f.score_samples(te[cols])
        out.append((roc_auc_score(te.y, score), average_precision_score(te.y, score)))
    return np.mean(out, axis=0)

## 1. Every feature, benign against tunnel

Median, 95th percentile and maximum per label, for the default set and the ablation columns.

In [ ]:
summary = m.groupby("label")[FEATS].agg(["median", lambda s: s.quantile(.95), "max"])
summary.columns = [f"{c} {s if isinstance(s, str) else 'p95'}" for c, s in summary.columns]
summary.T.round(2)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
for ax, col in zip(axes.ravel(), FEATS):
    for lab, color in [("benign", "tab:blue"), ("tunnel", "tab:red")]:
        ax.hist(m.loc[m.label == lab, col], bins=30, alpha=.6, color=color, label=lab, density=True)
    tag = "default" if col in DEFAULT else ("optional" if col == "data_qtype" else "ablation")
    ax.set_title(f"{col} ({tag})", fontsize=10); ax.set_yscale("log")
axes[0, 0].legend(); plt.tight_layout(); plt.savefig(FIG / "feature_distributions.png", dpi=150); plt.show()

## 2. Per-query shape against domain context, by tunnel phase

173 of the 222 tunnel queries are pings with short names. Per phase, the shape features of a ping look normal, while the context features still separate it.

In [ ]:
by_phase = m.groupby("phase")[FEATS].median().round(2)
by_phase["queries"] = m.groupby("phase").size()
by_phase.loc[["benign", "ping", "data", "size_probe", "handshake", "encoding_test"]]

In [ ]:
# How many tunnel queries would a simple threshold on one feature catch? (illustration only, not the detector)
tun = m[m.y == 1]
ben = m[m.y == 0]
rows = []
for col, thr in [("entropy", 4.0), ("name_bytes", 63), ("uniq_names_60s", 30), ("uniq_ratio_60s", 0.99)]:
    rows.append({"feature": col, "threshold": thr, "tunnel rows above": int((tun[col] > thr).sum()),
                 "benign rows above": int((ben[col] > thr).sum())})
pd.DataFrame(rows)

## 3. Benign look-alikes

The benign queries that score highest on each feature. These are the likely false positives.

In [ ]:
show = ["parent", "name", "name_bytes", "entropy", "uniq_names_60s", "uniq_ratio_60s", "norepeat_tol"]
prep = prepare(load_raw(), session="second")
mm = m.assign(name=prep["name"].to_numpy())
b = mm[mm.y == 0]
for col in ["uniq_names_60s", "entropy", "name_bytes", "norepeat_tol"]:
    print(f"--- highest benign {col}")
    display(b.nlargest(5, col)[show].assign(name=lambda d: d["name"].str[:60]))

## 4. The start of the tunnel session

Context features need evidence to build up. The first queries of the session look like a new benign domain.

In [ ]:
first_rows = m[m.y == 1].head(12)[["phase", "name_bytes", "uniq_names_60s", "uniq_ratio_60s", "norepeat_tol", "norepeat_evidence"]]
first_rows.round(2)

In [ ]:
# Timeline of uniq_names_60s: tunnel session against the busiest benign domains
fig, ax = plt.subplots(figsize=(11, 3.2))
m["dt"] = pd.to_datetime(m["ts"], unit="s")
ax.scatter(m.loc[m.y == 0, "dt"], m.loc[m.y == 0, "uniq_names_60s"], s=4, color="tab:blue", label="benign")
ax.scatter(m.loc[m.y == 1, "dt"], m.loc[m.y == 1, "uniq_names_60s"], s=4, color="tab:red", label="tunnel")
ax.axvline(pd.Timestamp("2023-01-07 18:30"), color="gray", ls="--"); ax.text(pd.Timestamp("2023-01-07 18:30:30"), 150, "train | test")
ax.set_ylabel("uniq_names_60s"); ax.legend(); plt.tight_layout(); plt.savefig(FIG / "uniq_names_timeline.png", dpi=150); plt.show()

## 5. Drift: how far does the test period move away from training?

Share of test benign rows above the largest training value, per feature. A feature with a large share makes the model flag normal traffic only because the load changed, not because of a tunnel.

In [ ]:
print("default run: session=second, cut 18:30")
display(drift_table(m, FEATS))
print("strict run: session=first, cut 18:04:36 (training = the quiet start only)")
print(m_first.groupby(["split", "label"]).size().unstack(fill_value=0))
display(drift_table(m_first, FEATS))

In [ ]:
# Do the same benign domains appear before and after the cut?
tr, te = m[m.split == "train"], m[(m.split == "test") & (m.y == 0)]
print("benign parent domains in train:", tr.parent.nunique(), "| in test:", te.parent.nunique(),
      "| in both:", len(set(tr.parent) & set(te.parent)))

## 6. Redundancy: which features correlate?

Spearman correlation on the benign training rows, i.e. on what the model learns "normal" from. `data_qtype` is constant (0) there and left out. Strongly correlated features carry the same information, and Isolation Forest picks a feature at random at every split, so a redundant block takes a large share of the splits.

In [ ]:
cols = [c for c in FEATS if c != "data_qtype"]
train_benign = m[m.split == "train"]
corr = train_benign[cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax)
ax.set_title("Spearman correlation, benign training rows")
plt.tight_layout(); plt.savefig(FIG / "feature_correlation.png", dpi=150); plt.show()
pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(cols) for b in cols[i + 1:] if abs(corr.loc[a, b]) >= 0.7]
print("pairs with |rho| >= 0.7:", pairs)

## 7. Which feature set ranks the tunnel best?

Isolation Forest trained on the benign training rows (300 trees, 10 seeds), scored on the test rows. Threshold-free ROC AUC and PR AUC, so no threshold is tuned on the test set. One tunnel event, so read the differences as direction, not as precise numbers. The strict run (training = 244 quiet rows) replays the same tunnel session, so it is a drift stress test, not an independent second result.

In [ ]:
ctx = fs["context"]
sets = {
    "shape only (name_bytes)": fs["shape"],
    "context only": ctx,
    "DEFAULT: name_bytes + context": DEFAULT,
    "default + entropy": DEFAULT + ["entropy"],
    "default + longest_label": DEFAULT + ["longest_label"],
    "all three shape + context": ["name_bytes", "longest_label", "entropy"] + ctx,
    "default minus norepeat_tol": [c for c in DEFAULT if c != "norepeat_tol"],
    "default + norepeat_evidence": DEFAULT + ["norepeat_evidence"],
    "default + log_cum_uniq": DEFAULT + ["log_cum_uniq"],
    "default + data_qtype": fs["full"],
    "all columns": FEATS,
}
rows = []
for name, cols in sets.items():
    p = rank_scores(m, cols)
    s = rank_scores(m_first, cols)
    rows.append({"feature set": name, "primary ROC": p[0], "primary PR": p[1], "strict ROC": s[0], "strict PR": s[1]})
ablation = pd.DataFrame(rows).set_index("feature set").round(3)
ablation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ablation[["primary PR", "strict PR"]].plot.barh(ax=ax)
ax.set_xlabel("PR AUC (test rows, 10 seeds)"); ax.invert_yaxis(); ax.set_ylabel("")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.savefig(FIG / "feature_set_comparison.png", dpi=150); plt.show()

## 8. `norepeat_evidence` against `norepeat_tol` when a tunnel name is repeated

`norepeat_evidence` is `log(1 + distinct names so far)` and becomes 0 for good after the first repeated name. `norepeat_tol` scales the same log by the share of queries so far that were new names, so one repeat barely changes it. `inject_repeats` repeats one tunnel query (a retransmission, or an attacker repeating a name on purpose).

In [ ]:
old = [c for c in DEFAULT if c != "norepeat_tol"] + ["norepeat_evidence"]
rows = []
frames = {}
for n in [0, 1, 5]:
    p = prep if n == 0 else inject_repeats(prep, n=n, at=30)
    frames[n] = build_features(p, 60)
    for name, cols in [("norepeat_tol (default)", DEFAULT), ("norepeat_evidence", old)]:
        r = rank_scores(frames[n], cols)
        rows.append({"repeated names": n, "feature": name, "ROC AUC": r[0], "PR AUC": r[1]})
display(pd.DataFrame(rows).round(3))

t = frames[1][frames[1].y == 1].sort_values("ts").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t.index, t["norepeat_evidence"], label="norepeat_evidence")
ax.plot(t.index, t["norepeat_tol"], label="norepeat_tol")
ax.set_xlabel("tunnel query number (one name repeated at query 30)"); ax.legend(); plt.tight_layout()
plt.savefig(FIG / "norepeat_after_repeat.png", dpi=150); plt.show()

## 9. Stress tests with simulated variants (`src/variants.py`)

Built from the one capture we have, so they test robustness but do not prove generalisation.

**9a. Slow tunnel.** The tunnel session stretched in time by a factor k (k = 60 is about 25 minutes, roughly 9 queries a minute). The names stay unique.

In [ ]:
rows = []
for k in [1, 5, 20, 60]:
    p = slow_tunnel(prep, k=k)
    for w in [60, 300]:
        fw = feature_sets(w)
        mw = build_features(p, w)
        for name, cols in [("context only", fw["context"]), ("default (name_bytes + context)", fw["forced"]),
                           ("all three shape + context", ["name_bytes", "longest_label", "entropy"] + fw["context"])]:
            r = rank_scores(mw, cols, seeds=5)
            rows.append({"slowdown": k, "window": f"{w}s", "feature set": name, "ROC": r[0], "PR": r[1]})
slow = pd.DataFrame(rows).round(3)
slow.pivot_table(index=["slowdown", "window"], columns="feature set", values=["ROC", "PR"])

**9b. Noisier network.** The benign traffic copied onto more hosts (shifted in time; half of each copy's names made host-specific, as CDN and session names are). The tunnel stays one host. "per client + parent" is the default counting; "per parent only" counts as before the grouping change.

In [ ]:
rows = []
for n_hosts in [1, 5, 10, 20]:
    for grouping in ["per client + parent", "per parent only"]:
        acc = {}
        for sim in ([0] if n_hosts == 1 else [0, 1]):
            v = prep if n_hosts == 1 else add_benign_clients(prep, n_extra=n_hosts - 1, seed=sim)
            if grouping == "per parent only":
                v = collapse_clients(v)
            mv = build_features(v, 60)
            for name, cols in [("context only", ctx), ("default (name_bytes + context)", DEFAULT),
                               ("all three shape + context", ["name_bytes", "longest_label", "entropy"] + ctx)]:
                acc.setdefault(name, []).append(rank_scores(mv, cols, seeds=3))
        for name, vals in acc.items():
            a = np.mean(vals, axis=0)
            rows.append({"hosts": n_hosts, "counting": grouping, "feature set": name, "ROC": a[0], "PR": a[1]})
noisy = pd.DataFrame(rows).round(3)
display(noisy[noisy.counting == "per client + parent"].pivot_table(index="hosts", columns="feature set", values="PR"))
display(noisy.pivot_table(index=["hosts", "counting"], columns="feature set", values=["ROC", "PR"]))

In [ ]:
pc = noisy[noisy.counting == "per client + parent"].pivot_table(index="hosts", columns="feature set", values="PR")
fig, ax = plt.subplots(figsize=(7, 4))
pc.plot(marker="o", ax=ax); ax.set_ylabel("PR AUC (per client + parent counting)"); ax.set_xlabel("benign hosts"); ax.set_ylim(0.6, 1.0)
plt.tight_layout(); plt.savefig(FIG / "noisy_hosts.png", dpi=150); plt.show()

## Observations

**What the features look like (sections 1 to 5)**

- **Shape alone does not separate the classes:** medians of benign and tunnel are almost equal (name length 20 / 19, entropy 2.50 / 2.75). Only the data and probe queries stand out. By phase, pings have median name length 19; data queries 93.5 bytes and 5.56 bits of entropy. `name_bytes` alone ranks worse than chance on the test rows (ROC 0.24), because 173 of the 222 tunnel queries are short pings.
- **Context features separate the classes:** `uniq_names_60s` median 1 / 111.5 (max 28 / 222), `uniq_ratio_60s` median 0.40 / 1.0, `norepeat_tol` median 0.40 / 4.72. Pings have a median `uniq_names_60s` of 133, so the context features catch the queries the shape features miss.
- **Simple thresholds (illustration, not the detector):** `entropy > 4` catches 28 tunnel rows but flags 201 benign rows; `name_bytes > 63` catches 38 and flags 16; `uniq_names_60s > 30` catches 192 and flags 0; `uniq_ratio_60s > 0.99` flags 689 benign rows, so the ratio is only useful combined with the count.
- **Benign look-alikes:** `yahoo.com` has the highest `uniq_names_60s` (28). Highest entropy: `akamaihd.net`, `go.com` and `akadns.net` names (up to 4.53). The longest benign name is an `office.net` name (103 bytes). Highest `norepeat_tol` are `azure.com` names (2.05), far below the tunnel median of 4.72.
- **The start of the session:** the first tunnel queries look like a new benign domain (`uniq_names_60s` 1, 2, 3 ... and `norepeat_tol` 0.69, 1.10, 1.39 ...). Evidence builds up over the first dozen queries.
- **Drift, default run:** share of test benign rows above the largest training value: `uniq_names_60s` 3.2%, `log_cum_uniq` 7.6%, `longest_label` 0.7%, `norepeat_tol` 0.5%, `entropy` 0.2%, `name_bytes` 0%. The strict run (244 quiet training rows) drifts more: 17.1% for `uniq_names_60s` and 19.6% for `log_cum_uniq`. Only 88 of the 172 benign parent domains seen in training also appear in the test period, so domain identity cannot be a feature.

**What the decisions rest on (sections 6 to 9)**

- **Redundancy:** only `name_bytes` with `longest_label` and with `entropy` correlate strongly on benign training rows (Spearman 0.80). The context features are only moderately related (below 0.7), and shape and context are nearly independent.
- **More shape features rank worse:** adding `entropy`, then `longest_label`, then both to the default lowers strict PR AUC from 0.974 to 0.870, 0.809 and 0.658. Isolation Forest picks features at random, so the redundant block takes more of the splits. The primary run shows the same direction (PR 0.993 to 0.975).
- **`norepeat_tol` carries the ranking:** without it, primary PR falls from 0.993 to 0.969 and strict PR from 0.974 to 0.533.
- **`norepeat_evidence` is slightly better on clean data** (PR 0.997 primary, 0.991 strict) but is switched off by one repeated name: with one repeat, PR falls to 0.956, while `norepeat_tol` stays at 0.993. That is the reason for the default.
- **`log_cum_uniq` drifts and lowers the strict result** (PR 0.887). `data_qtype` adds nothing (strict 0.961) and is iodine-specific.
- **Slow tunnels:** up to 60 times slower (about 25 minutes), every set keeps ROC 0.99 or higher. At the slowest setting with a 60 s window the default (PR 0.990) holds up better than context only (0.978) and three shape features (0.967). The long memory of `norepeat_tol` is what keeps slow tunnels detectable.
- **Noisier network (1 to 20 benign hosts):** the default stays at PR 0.986 to 0.993, while three shape features fall from 0.980 to 0.699. Counting per parent only did not do worse than per (client, parent) here, because the tunnel domain is only asked by the tunnel host. The per-client grouping is a design choice that this simulation does not confirm.

**Caveats**

- One tunnel event, one tool, one benign client; the strict run replays the same session. The simulations are built from this one capture and show robustness, not generalisation.
- The choice of `name_bytes` as the only shape feature rests on redundancy. The numbers are mixed: it helps at the slowest tunnel (0.990 against 0.978) but costs 0.011 PR in the strict run against context only (0.974 against 0.985). It is kept for coverage of the queries that carry data, not because these data demand it. Recheck it on long-name low-rate tunnels.
- Thresholds, false-positive analysis and event-level metrics are not done here; they belong to the modelling notebook.

## Decisions

| Feature | Decision | Reason | Evidence |
|---|---|---|---|
| `uniq_names_60s`, `uniq_ratio_60s` | **Default** | Caching forces a new name per query; hard for an attacker to avoid | Section 1; context only: PR 0.993 primary, 0.985 strict |
| `norepeat_tol` | **Default** | Long memory (slow tunnels) and survives a repeated name | Section 7: removing it gives strict PR 0.533; section 8: stable at 0.993 after repeats |
| `name_bytes` | **Default** (the only shape feature) | Data travels in the name; not tool-specific. One feature per correlated group | Sections 6, 7, 9a; mixed, see caveats |
| `longest_label`, `entropy` | Ablation | Redundant with `name_bytes` (rho 0.80) and crowd out the context features | Section 7: strict PR 0.809, 0.870, 0.658 when added |
| `norepeat_evidence` | Ablation | Best on clean data, but one repeated name switches it off for good | Section 8: PR 0.997 to 0.956 after one repeat |
| `log_cum_uniq` | Ablation | Grows with time since capture start, so it drifts | Section 5: 7.6% / 19.6% above train max; section 7: strict PR 0.887 |
| `data_qtype` | Optional | NULL is iodine's choice; a tool can switch to A or CNAME | Section 7: no gain (strict PR 0.961) |
| IPs, ports, AA, RA, TTL, EDNS0, `rtt`, NXDOMAIN, raw volume, domain identity | **Not features** | Separate the classes only because of the lab set-up | `docs/data_dictionary.md` |
